# Разведочный анализ цен подержанных автомобилей

Целевая переменная — `price`. Нужно понять, какие признаки связаны с ценой, где возможны проблемы с данными и насколько похожи обучающая и тестовая выборки. Наборы синтетические, поэтому наблюдения относятся к данным соревнования. [Описание данных Kaggle](https://www.kaggle.com/competitions/playground-series-s4e9/data)

Запустите ноутбук целиком с подключённым источником `playground-series-s4e9`. Здесь строятся графики и формулируются гипотезы для кросс-валидации.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

COMPETITION = "playground-series-s4e9"
TARGET = "price"
ID_COLUMN = "id"

data_dir = next(
    (path.parent for path in Path("/kaggle/input").rglob(f"{COMPETITION}/train.csv")),
    Path("data"),
)
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
features = test.columns.drop(ID_COLUMN)
train_features = train.loc[:, features]
test_features = test.loc[:, features]
price = train[TARGET]
splits = {"train": train_features, "test": test_features}

output_dir = Path("eda-output")
output_dir.mkdir(exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["svg.hashsalt"] = "used-car-prices-eda"
COLORS = {"train": "#266890", "test": "#cc652c"}

display(pd.DataFrame([train.shape, test.shape], index=splits, columns=["Строки", "Столбцы"]))
display(train.dtypes.rename("Тип").to_frame())


def show_figure(figure: plt.Figure, name: str) -> None:
    """Сохранить рисунок в SVG и показать его в ноутбуке."""
    figure.savefig(output_dir / f"{name}.svg", bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close(figure)


## Распределение цены и выбросы

Логарифмическая шкала помогает рассмотреть хвост распределения. Правило 1,5 межквартильного размаха отмечает необычные наблюдения для дальнейшего анализа. Порог сам по себе не обосновывает удаление строк. Для RMSE важно отдельно изучить самые дорогие автомобили, поскольку крупные ошибки сильнее влияют на метрику.

In [ ]:
quantiles = price.quantile([0, .01, .25, .5, .75, .95, .99, .999, 1])
q1, q3 = quantiles.loc[[.25, .75]]
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
price_summary = {
    "mean": float(price.mean()),
    "quantiles": {str(q): float(value) for q, value in quantiles.items()},
    "nonpositive": int(price.le(0).sum()),
    "lower_iqr_limit": float(lower),
    "upper_iqr_limit": float(upper),
    "below_lower_limit": int(price.lt(lower).sum()),
    "above_upper_limit": int(price.gt(upper).sum()),
}
display(quantiles.rename("Цена").to_frame())
display(train.nlargest(8, TARGET)[["brand", "model", "model_year", "milage", TARGET]])

positive = price[price.gt(0)]
body = price[price.le(quantiles.loc[.99])]
figure, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
axes[0].hist(body, bins=45, color=COLORS["train"], edgecolor="none")
axes[0].axvline(price.median(), linestyle="--", color="#983124", label="Медиана")
axes[0].set(title="Цены до 99-го процентиля", xlabel="Цена", ylabel="Число автомобилей")
axes[0].legend()
axes[1].hist(positive, bins=np.geomspace(positive.min(), positive.max(), 41),
             color=COLORS["train"], edgecolor="none")
axes[1].set(xscale="log", yscale="log", title="Все положительные цены",
            xlabel="Цена, логарифмическая шкала", ylabel="Частота, логарифмическая шкала")
show_figure(figure, "price-distribution")


## Пропуски, константы и повторы

Доли пропусков считаются отдельно для каждой выборки. Число уникальных значений не включает пропуски. Повторы признаков проверяются без `id`: разные идентификаторы могут иметь одинаковые описания. В таблице различаются полное совпадение признаков и идентичность всей обучающей записи без `id`.

In [ ]:
missing_pct = pd.DataFrame({
    name: frame.isna().mean().mul(100) for name, frame in splits.items()
})
unique_counts = pd.DataFrame({name: frame.nunique() for name, frame in splits.items()})
quality = pd.concat({"Пропуски, %": missing_pct, "Уникальные": unique_counts}, axis=1)
display(quality.round(3))

constant_features = {
    name: counts.index[counts.le(1)].tolist() for name, counts in unique_counts.items()
}
train_hash = pd.util.hash_pandas_object(train_features, index=False)
test_hash = pd.util.hash_pandas_object(test_features, index=False)
duplicates = {
    "train_repeated_features_after_first": int(train_features.duplicated().sum()),
    "test_repeated_features_after_first": int(test_features.duplicated().sum()),
    "train_repeated_features_and_price_after_first": int(
        train.drop(columns=ID_COLUMN).duplicated().sum()
    ),
    "test_rows_matching_train_features": int(test_hash.isin(train_hash).sum()),
}
display(pd.Series(duplicates, name="Строки").to_frame())

shown_missing = missing_pct.loc[missing_pct.gt(0).any(axis=1)].sort_values("train")
figure, ax = plt.subplots(figsize=(8.2, 3.7), layout="constrained")
shown_missing.plot.barh(ax=ax, color=[COLORS[name] for name in splits], width=.72)
ax.set(title="Пропуски в train и test", xlabel="Доля строк, %", ylabel="")
show_figure(figure, "missing-values")


## Связь цены с годом, пробегом и маркой

Для года выпуска и пробега покажем медиану цены и интервал от 25-го до 75-го процентиля. Пробег сгруппируем по квантилям, чтобы в каждом интервале было достаточно наблюдений. Марки сравним среди 12 самых частых. Различия между марками могут отражать состав модельного ряда, год выпуска и пробег.

In [ ]:
def grouped_prices(group: pd.Series) -> pd.DataFrame:
    """Посчитать размер группы, медиану и квартильный интервал цены."""
    return price.groupby(group, observed=True).agg(
        rows="size", median="median",
        q25=lambda values: values.quantile(.25),
        q75=lambda values: values.quantile(.75),
    )


year_price = grouped_prices(train["model_year"]).query("rows >= 100")
mileage_bins = pd.qcut(train["milage"], q=10, duplicates="drop")
mileage_price = grouped_prices(mileage_bins)
mileage_medians = train["milage"].groupby(mileage_bins, observed=True).median()
brand_price = grouped_prices(train["brand"]).nlargest(12, "rows").sort_values("median")

numeric = train_features.select_dtypes(include="number")
numeric_and_price = numeric.assign(price=price)
spearman = numeric_and_price.corr(method="spearman")
pearson = numeric_and_price.corr(method="pearson")
display(pd.concat({"Пирсон": pearson[TARGET], "Спирмен": spearman[TARGET]}, axis=1).round(3))
display(spearman.round(3))
display(brand_price.round(0))

figure, axes = plt.subplots(1, 3, figsize=(15, 4.6), layout="constrained")
for ax, (x, table, label) in zip(axes[:2], [
    (year_price.index, year_price, "Год выпуска"),
    (mileage_medians, mileage_price, "Пробег, мили"),
], strict=True):
    ax.plot(x, table["median"], color=COLORS["train"], marker="o", markersize=3)
    ax.fill_between(x, table["q25"], table["q75"], alpha=.21, color=COLORS["train"])
    ax.set(xlabel=label, ylabel="Цена", title=f"Цена и {label.lower()}")

axes[2].barh(brand_price.index, brand_price["median"], color=COLORS["train"])
axes[2].set(title="Медиана для 12 частых марок", xlabel="Цена")
show_figure(figure, "price-by-features")


## Сходство обучающей и тестовой выборок

Для числовых признаков сравним распределения на одинаковых интервалах. Для категориальных сравним доли значений, учитывая пропуски отдельно. Расстояние полной вариации равно половине суммы абсолютных различий долей. Ноль означает совпадение распределений. Доля незнакомых категорий показывает, сколько непустых значений в `test` отсутствуют в `train`, в процентах от всех тестовых строк.

In [ ]:
category_columns = features.difference(numeric.columns, sort=False)
records = []
for column in category_columns:
    train_values = train[column].astype("string")
    test_values = test[column].astype("string")
    frequencies = pd.concat(
        [
            values.fillna("<пропуск>").value_counts(normalize=True).rename(name)
            for values, name in [(train_values, "train"), (test_values, "test")]
        ],
        axis=1,
    ).fillna(0)
    unseen = test_values.notna() & ~test_values.isin(train_values.dropna())
    records.append({
        "feature": column,
        "distance_pct": float(frequencies["train"].sub(frequencies["test"]).abs().sum() * 50),
        "unseen_pct": float(unseen.mean() * 100),
    })

category_shift = pd.DataFrame(records).set_index("feature").sort_values("distance_pct")
display(category_shift.sort_values("distance_pct", ascending=False).round(3))
display(pd.concat({name: frame[numeric.columns].describe() for name, frame in splits.items()},
                  axis=1).round(1))

figure, axes = plt.subplots(1, 3, figsize=(15, 4.6), layout="constrained")
for ax, column, label in zip(
    axes[:2], ["model_year", "milage"], ["Год выпуска", "Пробег, мили"], strict=True
):
    pooled = pd.concat([frame[column] for frame in splits.values()]).dropna()
    edges = (
        np.arange(pooled.min() - .5, pooled.max() + 1.5)
        if column == "model_year"
        else np.histogram_bin_edges(pooled, bins=32)
    )
    for name, frame in splits.items():
        counts = np.histogram(frame[column].dropna(), bins=edges)[0]
        ax.stairs(counts / counts.sum() * 100, edges, label=name, color=COLORS[name], linewidth=1.5)
    ax.set(xlabel=label, ylabel="Доля строк в интервале, %", title=f"Train и test · {label.lower()}")
    ax.legend()
axes[2].barh(category_shift.index, category_shift["distance_pct"], color="#5c7987")
axes[2].set(title="Различия долей категорий", xlabel="Расстояние полной вариации, %")
show_figure(figure, "train-test-comparison")


## Выводы и гипотезы для следующих экспериментов

Числа ниже рассчитываются при каждом запуске. Они помогают выбрать эксперименты, а эффект каждого изменения сравним по RMSE на фиксированных фолдах.

In [ ]:
summary = {
    "rows": {"train": len(train), "test": len(test)},
    "price": price_summary,
    "missing_pct": missing_pct.loc[missing_pct.gt(0).any(axis=1)].to_dict(orient="index"),
    "constant_features": constant_features,
    "duplicates": duplicates,
    "spearman": {col: spearman[col].to_dict() for col in spearman.columns},
    "category_shift": category_shift.to_dict(orient="index"),
}
(output_dir / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("EDA_SUMMARY " + json.dumps(summary, ensure_ascii=False))

top_missing = missing_pct["train"].idxmax()
top_shift = category_shift["distance_pct"].idxmax()
unseen_most = category_shift["unseen_pct"].idxmax()
display(Markdown(f"""
- Медиана цены — **{price.median():,.0f}**, среднее — **{price.mean():,.0f}**.
  Максимум — **{price.max():,.0f}**. Выше верхней границы IQR
  **{price_summary['above_upper_limit']:,}** строк. Сравним обучение на исходной и
  логарифмированной цене, считая RMSE в исходных единицах.
- Больше всего пропусков в train у `{top_missing}` (**{missing_pct.at[top_missing, 'train']:.2f} %**).
  Проверим отдельную категорию для пропусков и сравним её с заполнением по группам.
  Число константных признаков — **{len(constant_features['train'])}** в train и
  **{len(constant_features['test'])}** в test.
- Корреляции Спирмена с ценой — **{spearman.at['model_year', TARGET]:.3f}** для года и
  **{spearman.at['milage', TARGET]:.3f}** для пробега. Проверим возраст машины,
  пробег на год и медианы по марке и модели. Последние будем считать внутри обучающего фолда.
- Наибольшее различие долей категорий у `{top_shift}` (**{category_shift.at[top_shift, 'distance_pct']:.2f} %**).
  Больше всего незнакомых значений у `{unseen_most}`
  (**{category_shift.at[unseen_most, 'unseen_pct']:.2f} %** строк test).
  Сравним кодирования, которые умеют обрабатывать значения, отсутствующие в обучении.
- В train **{duplicates['train_repeated_features_after_first']:,}** повторов набора признаков сверх
  первых записей. На этапе валидации проверим, попадают ли такие строки в разные фолды
  и насколько это меняет оценку.
"""))
